In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
#df=pd.read_csv("/content/machine-1-1.txt",header=None)

FileNotFoundError: [Errno 2] No such file or directory: '/content/machine-1-1.txt'

In [ ]:
#df.shape

In [ ]:
#df.head()

In [ ]:
import pandas as pd
import numpy as np

paths = {
    "machine-1-1": r"OmniAnomaly-master\ServerMachineDataset\train\machine-1-1.txt",   # adjust to your actual paths
    "machine-1-2": r"OmniAnomaly-master\ServerMachineDataset\train\machine-1-2.txt",
    "machine-1-3": r"OmniAnomaly-master\ServerMachineDataset\train\machine-1-3.txt",
}

cols = [f"metric_{i}" for i in range(38)]
data = {}
for name, p in paths.items():
    df = pd.read_csv(p, header=None, names=cols)
    data[name] = df
    print(f"{name}: shape={df.shape}")

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
def inspect(name, df):
    print(f"\n===== {name} =====")
    print("Shape:", df.shape)
    print("Dtypes:", df.dtypes.value_counts().to_dict())
    print("Missing values:", df.isna().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())
    print("Constant columns:", [c for c in df.columns if df[c].nunique() == 1])
    print("Global min/max:", df.min().min(), df.max().max())

for name, df in data.items():
    inspect(name, df)

# Per-metric statistics for one machine
data["machine-1-1"].describe().T

In [ ]:
summary = []
for name, df in data.items():
    summary.append({
        "machine": name,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "missing": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "constant_cols": [c for c in df.columns if df[c].nunique() == 1],
        "n_constant": sum(df[c].nunique() == 1 for c in df.columns),
        "min": df.min().min(),
        "max": df.max().max(),
        "non_numeric_cols": int((~df.dtypes.apply(pd.api.types.is_numeric_dtype)).sum()),
    })

summary_df = pd.DataFrame(summary).set_index("machine")
summary_df

In [ ]:
for name, df in data.items():
    print(f"\n===== {name} =====")
    display(df.describe().T.head(10))   # peek at the first 10 metrics

In [ ]:
const = {name: {c for c in df.columns if df[c].nunique() == 1} for name, df in data.items()}

for name, s in const.items():
    print(name, sorted(s, key=lambda x: int(x.split("_")[1])))

print("\nConstant in ALL machines:", sorted(set.intersection(*const.values()), key=lambda x: int(x.split("_")[1])))
print("Constant in ANY machine:", sorted(set.union(*const.values()), key=lambda x: int(x.split("_")[1])))

In [ ]:
for name, df in data.items():
    nunique = df.nunique().sort_values()
    print(f"\n{name}: lowest unique counts")
    print(nunique.head(12).to_string())

In [ ]:
def group_metrics(df, low_thresh=20):
    nun = df.nunique()
    dead = nun[nun == 1].index.tolist()
    low  = nun[(nun > 1) & (nun <= low_thresh)].index.tolist()
    cont = nun[nun > low_thresh].index.tolist()
    return dead, low, cont

groups = {}
for name, df in data.items():
    dead, low, cont = group_metrics(df)
    groups[name] = {"dead": dead, "low": low, "cont": cont}
    print(f"{name}: dead={len(dead)}, low-card={len(low)}, continuous={len(cont)}")

# What values do the low-cardinality metrics actually take?
for name, df in data.items():
    print(f"\n{name}")
    for c in groups[name]["low"]:
        print(c, sorted(df[c].round(4).unique().tolist()))

In [ ]:
# 1) How often does each level occur in the low-card metrics?
for name, df in data.items():
    print(f"\n===== {name} =====")
    for c in groups[name]["low"]:
        vc = df[c].round(4).value_counts(normalize=True).sort_index()
        print(f"\n{c}")
        print((vc * 100).round(3).to_string())

# 2) What value do the dead columns hold?
for name, df in data.items():
    d = groups[name]["dead"]
    print(name, df[d].iloc[0].round(4).to_dict())

# 3) Per-metric range for continuous metrics (is the train max really 1.0?)
for name, df in data.items():
    c = groups[name]["cont"]
    r = df[c].agg(["min", "max", "mean", "std"]).T.round(4)
    print(f"\n{name}: continuous metrics with max < 0.5")
    print(r[r["max"] < 0.5].to_string())

In [ ]:
import matplotlib.pyplot as plt

def dist_summary(df, cols):
    x = df[cols]
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    iqr = q3 - q1
    out = pd.DataFrame({
        "min": x.min(), "q01": x.quantile(0.01), "median": x.median(),
        "mean": x.mean(), "q99": x.quantile(0.99), "max": x.max(),
        "std": x.std(), "skew": x.skew(), "kurt": x.kurt(),
        "pct_zero": (x == 0).mean() * 100,
        "pct_outside_IQR": ((x < q1 - 1.5*iqr) | (x > q3 + 1.5*iqr)).mean() * 100,
    })
    return out.round(4)

for name, df in data.items():
    print(f"\n===== {name} (continuous, sorted by skew) =====")
    print(dist_summary(df, groups[name]["cont"]).sort_values("skew", ascending=False).to_string())

# Histograms (log y-axis so rare tails are visible)
for name, df in data.items():
    cols = groups[name]["cont"]
    nrows = int(np.ceil(len(cols) / 5))
    fig, axes = plt.subplots(nrows, 5, figsize=(20, 3 * nrows))
    for ax, c in zip(axes.ravel(), cols):
        ax.hist(df[c], bins=50)
        ax.set_yscale("log")
        ax.set_title(c, fontsize=9)
    for ax in axes.ravel()[len(cols):]:
        ax.axis("off")
    fig.suptitle(name)
    plt.tight_layout()
    plt.show()

# Exact duplicate columns among non-dead metrics
for name, df in data.items():
    live = df.loc[:, df.nunique() > 1]
    dup = live.columns[live.T.duplicated(keep=False)].tolist()
    print(name, "identical columns:", dup)

In [ ]:
# 1) Are extreme rows shared across metrics? (per machine, continuous metrics only)
for name, df in data.items():
    cols = groups[name]["cont"]
    x = df[cols]
    q99 = x.quantile(0.99)
    extreme = (x > q99)                       # per-metric top 1%
    per_row = extreme.sum(axis=1)
    print(f"\n{name}: rows where >=5 metrics are in their top 1%: {(per_row >= 5).sum()}")
    print("top 10 rows by number of extreme metrics:")
    print(per_row.sort_values(ascending=False).head(10).to_string())

# 2) Rows where many metrics hit exactly 0 (machine-1-1 suspicion)
for name, df in data.items():
    cols = groups[name]["cont"]
    z = (df[cols] == 0).sum(axis=1)
    print(f"\n{name}: rows with >=8 continuous metrics equal to 0:", (z >= 8).sum())
    print(z.sort_values(ascending=False).head(5).to_string())

# 3) Side-by-side comparison of shared continuous metrics (median, q99, std)
shared = sorted(set.intersection(*[set(g["cont"]) for g in groups.values()]),
                key=lambda s: int(s.split("_")[1]))
cmp = pd.concat({name: df[shared].agg(["median", "std"]).T.round(4)
                 for name, df in data.items()}, axis=1)
print("\nShared continuous metrics:", len(shared))
print(cmp.to_string())

# 4) Plot a few metrics over time for each machine
fig, axes = plt.subplots(3, 3, figsize=(18, 9))
for i, (name, df) in enumerate(data.items()):
    for j, c in enumerate(["metric_0", "metric_5", "metric_9"]):
        axes[i, j].plot(df[c].values, lw=0.4)
        axes[i, j].set_title(f"{name} {c}", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# 1) Are the machine-1-2 zero rows contiguous (dropout) or scattered?
df = data["machine-1-2"]; cols = groups["machine-1-2"]["cont"]
z_rows = (df[cols] == 0).sum(axis=1) >= 8
idx = np.where(z_rows)[0]
runs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
print("machine-1-2 zero-rows:", len(idx), "in", len(runs), "runs; longest:", max(len(r) for r in runs))
print("first runs:", [(r[0], r[-1]) for r in runs[:10]])

# 2) Burst windows: contiguous runs of rows with >=8 metrics in top 1%
for name, d in data.items():
    c = groups[name]["cont"]
    ext = ((d[c] > d[c].quantile(0.99)).sum(axis=1) >= 8).values
    i = np.where(ext)[0]
    rs = np.split(i, np.where(np.diff(i) > 5)[0] + 1) if len(i) else []
    print(name, "burst windows:", [(r[0], r[-1]) for r in rs if len(r)])

# 3) Scaling check: spread of the live metrics, per machine
for name, d in data.items():
    live = d.loc[:, d.nunique() > 1]
    s = live.std()
    rng = live.quantile(0.99) - live.quantile(0.01)
    print(f"\n{name}: std min/median/max = {s.min():.4f} / {s.median():.4f} / {s.max():.4f}")
    print(f"  q99-q01 range min/median/max = {rng.min():.4f} / {rng.median():.4f} / {rng.max():.4f}")
    print("  narrowest metrics (q99-q01):", rng.sort_values().head(5).round(4).to_dict())

# 4) Correlations (Spearman, live metrics only, per machine)
for name, d in data.items():
    live = d.loc[:, d.nunique() > 1]
    corr = live.corr(method="spearman")
    pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), 1)).stack()
    high = pairs[pairs.abs() > 0.95].sort_values(ascending=False)
    print(f"\n{name}: {len(high)} pairs with |rho|>0.95 (of {len(pairs)})")
    print(high.head(15).round(3).to_string())
    print("weakest-connected metrics (mean |rho|):")
    print(corr.abs().mean().sort_values().head(5).round(3).to_dict())

# 5) Heatmaps
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for ax, (name, d) in zip(axes, data.items()):
    live = d.loc[:, d.nunique() > 1]
    ax.imshow(live.corr(method="spearman"), cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
for name, df in data.items():
    print(f"\n===== {name} =====")
    print("dtypes:", df.dtypes.unique())
    display(df.head())
    display(df.describe().T.round(4))

# Compare all 38 metrics side by side (mean and std per machine)
cmp_all = pd.concat({n: d.agg(["mean", "std"]).T.round(4) for n, d in data.items()}, axis=1)
display(cmp_all)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plot_metrics = ["metric_0", "metric_5", "metric_6", "metric_9", "metric_15", "metric_21"]

for name, df in data.items():
    fig, axes = plt.subplots(len(plot_metrics), 1, figsize=(18, 2.2 * len(plot_metrics)), sharex=True)
    for ax, c in zip(axes, plot_metrics):
        ax.plot(df[c].values, lw=0.4, color="tab:blue", label="raw")
        ax.plot(df[c].rolling(500, center=True).median().values, lw=1.2, color="tab:red", label="rolling median (500)")
        ax.set_ylabel(c, fontsize=9)
    axes[0].legend(loc="upper right", fontsize=8)
    axes[-1].set_xlabel("row index (time order)")
    fig.suptitle(f"{name}: selected metrics over time", y=1.0)
    plt.tight_layout()
    plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(18, 8))
for i, (name, df) in enumerate(data.items()):
    c = groups[name]["cont"]
    n_extreme = (df[c] > df[c].quantile(0.99)).sum(axis=1)
    n_zero = (df[c] == 0).sum(axis=1)
    axes[i, 0].plot(n_extreme.values, lw=0.4)
    axes[i, 0].set_title(f"{name}: # metrics in top 1% per row", fontsize=9)
    axes[i, 1].plot(n_zero.values, lw=0.4, color="tab:green")
    axes[i, 1].set_title(f"{name}: # continuous metrics equal to 0 per row", fontsize=9)
plt.tight_layout()
plt.show()

# Where do machine-1-2's zero rows start, and are they spread evenly?
d = data["machine-1-2"]; c = groups["machine-1-2"]["cont"]
z = np.where((d[c] == 0).sum(axis=1) >= 8)[0]
print("first / last zero-row:", z.min(), z.max())
print(pd.Series(z).groupby(pd.Series(z) // 2000 * 2000).size())   # count per 2000-row block

In [ ]:
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster
from scipy.spatial.distance import squareform

def clean_corr(df, top_freq_max=0.95, method="spearman"):
    # keep only metrics that really vary (one value in <95% of rows)
    keep = [c for c in df.columns
            if df[c].nunique() > 1 and df[c].value_counts(normalize=True).iloc[0] < top_freq_max]
    dropped = [c for c in df.columns if c not in keep]
    corr = df[keep].corr(method=method).fillna(0)
    return corr, dropped

def order_and_groups(corr, cut=0.10):
    # distance = 1 - |rho|; cut=0.10 means groups with average |rho| >= ~0.9
    dist = 1 - corr.abs().values
    np.fill_diagonal(dist, 0)
    dist = (dist + dist.T) / 2
    Z = linkage(squareform(dist, checks=False), method="average")
    return leaves_list(Z), fcluster(Z, t=cut, criterion="distance")

def plot_corr(name, df, cut=0.10):
    corr, dropped = clean_corr(df)
    order, labels = order_and_groups(corr, cut)
    co = corr.iloc[order, order]
    lab_o = labels[order]
    short = [c.replace("metric_", "") for c in co.columns]

    annot = np.where(co.abs().values >= 0.5, co.round(1).astype(str).values, "")
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(co, cmap="vlag", vmin=-1, vmax=1, center=0, square=True,
                xticklabels=short, yticklabels=short,
                annot=annot, fmt="", annot_kws={"fontsize": 6},
                linewidths=0.3, linecolor="white",
                cbar_kws={"shrink": 0.7, "label": "Spearman rho"}, ax=ax)
    # outline the groups
    for i in range(1, len(lab_o)):
        if lab_o[i] != lab_o[i - 1]:
            ax.axhline(i, color="black", lw=1)
            ax.axvline(i, color="black", lw=1)
    ax.set_title(f"{name}: {len(co)} varying metrics (clustered; "
                 f"excluded as near-constant: {len(dropped)})")
    plt.tight_layout()
    plt.show()

    print(f"{name} excluded (near-constant):", [d.replace('metric_','') for d in dropped])
    for g in sorted(set(labels)):
        members = [corr.columns[i].replace("metric_", "") for i in range(len(labels)) if labels[i] == g]
        if len(members) > 1:
            print(f"  group {g}: {members}")

for name, df in data.items():
    plot_corr(name, df)

In [ ]:
for name, df in data.items():
    corr, dropped = clean_corr(df)
    order, labels = order_and_groups(corr, cut=0.25)
    print(f"\n{name}")
    for g in sorted(set(labels)):
        members = [corr.columns[i].replace("metric_", "") for i in range(len(labels)) if labels[i] == g]
        print(f"  group {g}: {members}")

# **TEST**

In [ ]:
import pandas as pd
import numpy as np

paths = {
    "machine-1-1": r"OmniAnomaly-master\ServerMachineDataset\test\machine-1-1.txt",   # adjust to your actual paths
    "machine-1-2": r"OmniAnomaly-master\ServerMachineDataset\test\machine-1-2.txt",
    "machine-1-3": r"OmniAnomaly-master\ServerMachineDataset\test\machine-1-3.txt",
}

cols = [f"metric_{i}" for i in range(38)]
test = {}
for name, p in paths.items():
    pop = pd.read_csv(p, header=None, names=cols)
    data[name] = pop
    print(f"{name}: shape={pop.shape}")

In [ ]:
def inspect(name, pop):
    print(f"\n===== {name} =====")
    print("Shape:", pop.shape)
    print("Dtypes:", pop.dtypes.value_counts().to_dict())
    print("Missing values:", pop.isna().sum().sum())
    print("Duplicate rows:", pop.duplicated().sum())
    print("Constant columns:", [c for c in pop.columns if pop[c].nunique() == 1])
    print("Global min/max:", pop.min().min(), pop.max().max())

for name, pop in data.items():
    inspect(name, pop)

# Per-metric statistics for one machine
data["machine-1-1"].describe().T

In [ ]:
summary = []
for name, pop in data.items():
    summary.append({
        "machine": name,
        "rows": pop.shape[0],
        "cols": pop.shape[1],
        "missing": int(pop.isna().sum().sum()),
        "duplicate_rows": int(pop.duplicated().sum()),
        "constant_cols": [c for c in pop.columns if pop[c].nunique() == 1],
        "n_constant": sum(pop[c].nunique() == 1 for c in pop.columns),
        "min": pop.min().min(),
        "max": pop.max().max(),
        "non_numeric_cols": int((~pop.dtypes.apply(pd.api.types.is_numeric_dtype)).sum()),
    })

summary_df = pd.DataFrame(summary).set_index("machine")
summary_df

In [ ]:
for name, pop in data.items():
    print(f"\n===== {name} =====")
    display(pop.describe().T.head(10))   # peek at the first 10 metrics

In [ ]:
const = {name: {c for c in pop.columns if pop[c].nunique() == 1} for name, pop in data.items()}

for name, s in const.items():
    print(name, sorted(s, key=lambda x: int(x.split("_")[1])))

print("\nConstant in ALL machines:", sorted(set.intersection(*const.values()), key=lambda x: int(x.split("_")[1])))
print("Constant in ANY machine:", sorted(set.union(*const.values()), key=lambda x: int(x.split("_")[1])))

In [ ]:
# THIS BLOCK WAS JUST HERE TO CHECK WHETHER LIKE ARE WE BY MISTAKE USING THE TRAIN FILE AS TEST CAUSE THEY HAVE THE SAME NAME
#BEFORE ME CREATING A SEPERATE FOLDER

import numpy as np

train = np.loadtxt(r"OmniAnomaly-master\ServerMachineDataset\train\machine-1-1.txt", delimiter=",")
test = np.loadtxt(r"OmniAnomaly-master\ServerMachineDataset\test\machine-1-1.txt", delimiter=",")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Exactly identical:", np.array_equal(train, test))

In [ ]:
for name, pop in data.items():
    nunique = pop.nunique().sort_values()
    print(f"\n{name}: lowest unique counts")
    print(nunique.head(12).to_string())

In [ ]:
def group_metrics(pop, low_thresh=20):
    nun = pop.nunique()
    dead = nun[nun == 1].index.tolist()
    low  = nun[(nun > 1) & (nun <= low_thresh)].index.tolist()
    cont = nun[nun > low_thresh].index.tolist()
    return dead, low, cont

groups = {}
for name, pop in data.items():
    dead, low, cont = group_metrics(pop)
    groups[name] = {"dead": dead, "low": low, "cont": cont}
    print(f"{name}: dead={len(dead)}, low-card={len(low)}, continuous={len(cont)}")

# What values do the low-cardinality metrics actually take?
for name, pop in data.items():
    print(f"\n{name}")
    for c in groups[name]["low"]:
        print(c, sorted(pop[c].round(4).unique().tolist()))

In [ ]:
# 1) How often does each level occur in the low-card metrics?
for name, pop in data.items():
    print(f"\n===== {name} =====")
    for c in groups[name]["low"]:
        vc = pop[c].round(4).value_counts(normalize=True).sort_index()
        print(f"\n{c}")
        print((vc * 100).round(3).to_string())

# 2) What value do the dead columns hold?
for name, pop in data.items():
    d = groups[name]["dead"]
    print(name, pop[d].iloc[0].round(4).to_dict())

# 3) Per-metric range for continuous metrics (is the train max really 1.0?)
for name, pop in data.items():
    c = groups[name]["cont"]
    r = pop[c].agg(["min", "max", "mean", "std"]).T.round(4)
    print(f"\n{name}: continuous metrics with max < 0.5")
    print(r[r["max"] < 0.5].to_string())

In [ ]:
import matplotlib.pyplot as plt

def dist_summary(pop, cols):
    x = pop[cols]
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    iqr = q3 - q1
    out = pd.DataFrame({
        "min": x.min(), "q01": x.quantile(0.01), "median": x.median(),
        "mean": x.mean(), "q99": x.quantile(0.99), "max": x.max(),
        "std": x.std(), "skew": x.skew(), "kurt": x.kurt(),
        "pct_zero": (x == 0).mean() * 100,
        "pct_outside_IQR": ((x < q1 - 1.5*iqr) | (x > q3 + 1.5*iqr)).mean() * 100,
    })
    return out.round(4)

for name, pop in data.items():
    print(f"\n===== {name} (continuous, sorted by skew) =====")
    print(dist_summary(pop, groups[name]["cont"]).sort_values("skew", ascending=False).to_string())

# Histograms (log y-axis so rare tails are visible)
for name, pop in data.items():
    cols = groups[name]["cont"]
    nrows = int(np.ceil(len(cols) / 5))
    fig, axes = plt.subplots(nrows, 5, figsize=(20, 3 * nrows))
    for ax, c in zip(axes.ravel(), cols):
        ax.hist(pop[c], bins=50)
        ax.set_yscale("log")
        ax.set_title(c, fontsize=9)
    for ax in axes.ravel()[len(cols):]:
        ax.axis("off")
    fig.suptitle(name)
    plt.tight_layout()
    plt.show()

# Exact duplicate columns among non-dead metrics
for name, pop in data.items():
    live = pop.loc[:, pop.nunique() > 1]
    dup = live.columns[live.T.duplicated(keep=False)].tolist()
    print(name, "identical columns:", dup)

In [ ]:
# 1) Are extreme rows shared across metrics? (per machine, continuous metrics only)
for name, pop in data.items():
    cols = groups[name]["cont"]
    x = pop[cols]
    q99 = x.quantile(0.99)
    extreme = (x > q99)                       # per-metric top 1%
    per_row = extreme.sum(axis=1)
    print(f"\n{name}: rows where >=5 metrics are in their top 1%: {(per_row >= 5).sum()}")
    print("top 10 rows by number of extreme metrics:")
    print(per_row.sort_values(ascending=False).head(10).to_string())

# 2) Rows where many metrics hit exactly 0 (machine-1-1 suspicion)
for name, pop in data.items():
    cols = groups[name]["cont"]
    z = (pop[cols] == 0).sum(axis=1)
    print(f"\n{name}: rows with >=8 continuous metrics equal to 0:", (z >= 8).sum())
    print(z.sort_values(ascending=False).head(5).to_string())

# 3) Side-by-side comparison of shared continuous metrics (median, q99, std)
shared = sorted(set.intersection(*[set(g["cont"]) for g in groups.values()]),
                key=lambda s: int(s.split("_")[1]))
cmp = pd.concat({name: pop[shared].agg(["median", "std"]).T.round(4)
                 for name, pop in data.items()}, axis=1)
print("\nShared continuous metrics:", len(shared))
print(cmp.to_string())

# 4) Plot a few metrics over time for each machine
fig, axes = plt.subplots(3, 3, figsize=(18, 9))
for i, (name, pop) in enumerate(data.items()):
    for j, c in enumerate(["metric_0", "metric_5", "metric_9"]):
        axes[i, j].plot(pop[c].values, lw=0.4)
        axes[i, j].set_title(f"{name} {c}", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# 1) Are the machine-1-2 zero rows contiguous (dropout) or scattered?
pop = data["machine-1-2"]; cols = groups["machine-1-2"]["cont"]
z_rows = (pop[cols] == 0).sum(axis=1) >= 8
idx = np.where(z_rows)[0]
runs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
print("machine-1-2 zero-rows:", len(idx), "in", len(runs), "runs; longest:", max(len(r) for r in runs))
print("first runs:", [(r[0], r[-1]) for r in runs[:10]])

# 2) Burst windows: contiguous runs of rows with >=8 metrics in top 1%
for name, d in data.items():
    c = groups[name]["cont"]
    ext = ((d[c] > d[c].quantile(0.99)).sum(axis=1) >= 8).values
    i = np.where(ext)[0]
    rs = np.split(i, np.where(np.diff(i) > 5)[0] + 1) if len(i) else []
    print(name, "burst windows:", [(r[0], r[-1]) for r in rs if len(r)])

# 3) Scaling check: spread of the live metrics, per machine
for name, d in data.items():
    live = d.loc[:, d.nunique() > 1]
    s = live.std()
    rng = live.quantile(0.99) - live.quantile(0.01)
    print(f"\n{name}: std min/median/max = {s.min():.4f} / {s.median():.4f} / {s.max():.4f}")
    print(f"  q99-q01 range min/median/max = {rng.min():.4f} / {rng.median():.4f} / {rng.max():.4f}")
    print("  narrowest metrics (q99-q01):", rng.sort_values().head(5).round(4).to_dict())

# 4) Correlations (Spearman, live metrics only, per machine)
for name, d in data.items():
    live = d.loc[:, d.nunique() > 1]
    corr = live.corr(method="spearman")
    pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), 1)).stack()
    high = pairs[pairs.abs() > 0.95].sort_values(ascending=False)
    print(f"\n{name}: {len(high)} pairs with |rho|>0.95 (of {len(pairs)})")
    print(high.head(15).round(3).to_string())
    print("weakest-connected metrics (mean |rho|):")
    print(corr.abs().mean().sort_values().head(5).round(3).to_dict())

# 5) Heatmaps
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for ax, (name, d) in zip(axes, data.items()):
    live = d.loc[:, d.nunique() > 1]
    ax.imshow(live.corr(method="spearman"), cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
for name, pop in data.items():
    print(f"\n===== {name} =====")
    print("dtypes:", pop.dtypes.unique())
    display(pop.head())
    display(pop.describe().T.round(4))

# Compare all 38 metrics side by side (mean and std per machine)
cmp_all = pd.concat({n: d.agg(["mean", "std"]).T.round(4) for n, d in data.items()}, axis=1)
display(cmp_all)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plot_metrics = ["metric_0", "metric_5", "metric_6", "metric_9", "metric_15", "metric_21"]

for name, pop in data.items():
    fig, axes = plt.subplots(len(plot_metrics), 1, figsize=(18, 2.2 * len(plot_metrics)), sharex=True)
    for ax, c in zip(axes, plot_metrics):
        ax.plot(pop[c].values, lw=0.4, color="tab:blue", label="raw")
        ax.plot(pop[c].rolling(500, center=True).median().values, lw=1.2, color="tab:red", label="rolling median (500)")
        ax.set_ylabel(c, fontsize=9)
    axes[0].legend(loc="upper right", fontsize=8)
    axes[-1].set_xlabel("row index (time order)")
    fig.suptitle(f"{name}: selected metrics over time", y=1.0)
    plt.tight_layout()
    plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(18, 8))
for i, (name, pop) in enumerate(data.items()):
    c = groups[name]["cont"]
    n_extreme = (pop[c] > pop[c].quantile(0.99)).sum(axis=1)
    n_zero = (pop[c] == 0).sum(axis=1)
    axes[i, 0].plot(n_extreme.values, lw=0.4)
    axes[i, 0].set_title(f"{name}: # metrics in top 1% per row", fontsize=9)
    axes[i, 1].plot(n_zero.values, lw=0.4, color="tab:green")
    axes[i, 1].set_title(f"{name}: # continuous metrics equal to 0 per row", fontsize=9)
plt.tight_layout()
plt.show()

# Where do machine-1-2's zero rows start, and are they spread evenly?
d = data["machine-1-2"]; c = groups["machine-1-2"]["cont"]
z = np.where((d[c] == 0).sum(axis=1) >= 8)[0]
print("first / last zero-row:", z.min(), z.max())
print(pd.Series(z).groupby(pd.Series(z) // 2000 * 2000).size())   # count per 2000-row block

In [ ]:
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster
from scipy.spatial.distance import squareform

def clean_corr(pop, top_freq_max=0.95, method="spearman"):
    # keep only metrics that really vary (one value in <95% of rows)
    keep = [c for c in pop.columns
            if pop[c].nunique() > 1 and pop[c].value_counts(normalize=True).iloc[0] < top_freq_max]
    dropped = [c for c in pop.columns if c not in keep]
    corr = pop[keep].corr(method=method).fillna(0)
    return corr, dropped

def order_and_groups(corr, cut=0.10):
    # distance = 1 - |rho|; cut=0.10 means groups with average |rho| >= ~0.9
    dist = 1 - corr.abs().values
    np.fill_diagonal(dist, 0)
    dist = (dist + dist.T) / 2
    Z = linkage(squareform(dist, checks=False), method="average")
    return leaves_list(Z), fcluster(Z, t=cut, criterion="distance")

def plot_corr(name, pop, cut=0.10):
    corr, dropped = clean_corr(pop)
    order, labels = order_and_groups(corr, cut)
    co = corr.iloc[order, order]
    lab_o = labels[order]
    short = [c.replace("metric_", "") for c in co.columns]

    annot = np.where(co.abs().values >= 0.5, co.round(1).astype(str).values, "")
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(co, cmap="vlag", vmin=-1, vmax=1, center=0, square=True,
                xticklabels=short, yticklabels=short,
                annot=annot, fmt="", annot_kws={"fontsize": 6},
                linewidths=0.3, linecolor="white",
                cbar_kws={"shrink": 0.7, "label": "Spearman rho"}, ax=ax)
    # outline the groups
    for i in range(1, len(lab_o)):
        if lab_o[i] != lab_o[i - 1]:
            ax.axhline(i, color="black", lw=1)
            ax.axvline(i, color="black", lw=1)
    ax.set_title(f"{name}: {len(co)} varying metrics (clustered; "
                 f"excluded as near-constant: {len(dropped)})")
    plt.tight_layout()
    plt.show()

    print(f"{name} excluded (near-constant):", [d.replace('metric_','') for d in dropped])
    for g in sorted(set(labels)):
        members = [corr.columns[i].replace("metric_", "") for i in range(len(labels)) if labels[i] == g]
        if len(members) > 1:
            print(f"  group {g}: {members}")

for name, pop in data.items():
    plot_corr(name, pop)

In [ ]:
for name, pop in data.items():
    corr, dropped = clean_corr(pop)
    order, labels = order_and_groups(corr, cut=0.25)
    print(f"\n{name}")
    for g in sorted(set(labels)):
        members = [corr.columns[i].replace("metric_", "") for i in range(len(labels)) if labels[i] == g]
        print(f"  group {g}: {members}")

In [ ]:
for var, val in list(globals().items()):
    if var.startswith("_"):
        continue
    if isinstance(val, pd.DataFrame):
        print(f"df   {var:10} shape={val.shape}  has metric_0: {'metric_0' in val.columns}  first cols: {val.columns[:3].tolist()}")
    elif isinstance(val, dict) and val and all(isinstance(v, pd.DataFrame) for v in val.values()):
        for k, v in val.items():
            print(f"dict {var:10}[{k}] shape={v.shape}  has metric_0: {'metric_0' in v.columns}  first cols: {v.columns[:3].tolist()}")

In [ ]:
print(pop.columns[:5].tolist() if isinstance(pop, pd.DataFrame) else {k: v.columns[:3].tolist() for k, v in pop.items()})